# Exploratory Analysis of LLM Prompts Configuration Sweeps

Comparing parameter estimations and social/personalized correlations across Qwen PCA embeddings and ST-PCA.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import json
import os

# 1. Locate the latest run directories
def get_latest_run(prefix):
    dirs = glob.glob(f"../results/llm/{prefix}-*")
    return max(dirs, key=os.path.getmtime) if dirs else None

qwen_dir = get_latest_run("colors_qwen_pca")
st_dir = get_latest_run("colors_st_pca")

def load_data(run_dir):
    if not run_dir: return None
    csv_path = os.path.join(run_dir, "aggregates", "criteria_curve_summary.csv")
    if os.path.exists(csv_path):
        return pd.read_csv(csv_path)
    return None

def load_checkpoints(run_dir):
    if not run_dir: return None
    outputs = glob.glob(os.path.join(run_dir, "outputs", "*.json"))
    if not outputs: return None
    with open(outputs[0], 'r') as f:
        data = json.load(f)
    return data

df_qwen = load_data(qwen_dir)
df_st = load_data(st_dir)
qwen_json = load_checkpoints(qwen_dir)

domain_colors = ["red", "blue", "green", "yellow", "purple"]


In [ ]:
# Figure 1: Social Kendall Correlation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Figure 1: Social Kendall Correlation (Qwen PCA vs SentenceTransformers PCA)")

for df, ax, title in [(df_qwen, ax1, "Qwen PCA"), (df_st, ax2, "SentenceTransformers PCA")]:
    if df is not None:
        for criterion in df['criterion'].unique():
            sub_df = df[df['criterion'] == criterion]
            ax.plot(sub_df['n_observations'], sub_df['social_tau_mean'], label=criterion)
        ax.set_title(title)
        ax.set_xlabel("Number of Observations")
        ax.set_ylabel("Social Kendall Tau")
        ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Figure 2: Personalized Choice Correlation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Figure 2: Personalized Choice Kendall Correlation (Mean)")

for df, ax, title in [(df_qwen, ax1, "Qwen PCA"), (df_st, ax2, "SentenceTransformers PCA")]:
    if df is not None and 'mean_person_tau_mean' in df.columns:
        for criterion in df['criterion'].unique():
            sub_df = df[df['criterion'] == criterion]
            ax.plot(sub_df['n_observations'], sub_df['mean_person_tau_mean'], label=criterion)
        ax.set_title(title)
        ax.set_xlabel("Number of Observations")
        ax.set_ylabel("Mean Personalized Kendall Tau")
        ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Figure 3: Final Delta and B matrix
if qwen_json and "checkpoints" in qwen_json and len(qwen_json["checkpoints"]) > 0:
    last_step = max([int(k) for k in qwen_json["checkpoints"].keys()])
    final_params = qwen_json["checkpoints"][str(last_step)]
    delta = np.array(final_params["delta"])
    B = np.array(final_params["interaction"])
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Figure 3: Final Delta Weights and Interaction Matrix B (Step {last_step})")
    
    bars = ax1.bar(domain_colors, delta, color=domain_colors, edgecolor='black')
    ax1.set_title("Base Utilities (Delta)")
    ax1.set_ylabel("Utility Value")
    
    cax = ax2.imshow(B, cmap="coolwarm", aspect="auto")
    ax2.set_title("Interaction Matrix B (Agent Features x Alternatives)")
    ax2.set_xlabel("Alternative Index")
    ax2.set_ylabel("Agent Feature Index")
    ax2.set_xticks(range(len(domain_colors)))
    ax2.set_xticklabels(domain_colors)
    fig.colorbar(cax, ax=ax2)
    
    plt.tight_layout()
    plt.show()
else:
    print("No checkpoints available to render Delta and B representations.")
